# Calibración de v2.10.0 con muestra real del RUES

Sprint 1.1 del plan de producción. Tres pasos:

1. **Instalar v2.10.0** desde el tarball en Drive.
2. **Correr el sistema sobre tu muestra real** y producir métricas globales sin ground truth (estadísticas de cluster, candidatos sospechosos).
3. **Generar 1500 pares estratificados para etiquetar manualmente**. Una vez los etiquetes en Excel/Sheets, en la siguiente sesión usas `medir_con_ground_truth.py` para tener el primer F1 real.

Tiempo esperado:
- Instalación + corrida sobre 50k regs: ~5-10 min.
- Etiquetado manual de 1500 pares: 8-12 horas tuyas (no en el notebook).
- Medición tras etiquetado: ~5 min.

## Paso 0 — Montar Drive y configurar rutas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# ── EDITA ESTAS RUTAS ANTES DE CORRER ──────────────────────────
# Ruta del tarball de la librería (subido a Drive)
TARBALL = Path('/content/drive/MyDrive/rues-linker-v2.10.0.tar.gz')

# Tu muestra real del RUES (CSV o parquet, con columnas NIT y RAZON_SOCIAL)
MUESTRA_REAL = Path('/content/drive/MyDrive/rues_muestra.csv')
COL_NIT = 'NIT'
COL_NAME = 'RAZON_SOCIAL'

# Carpeta donde se guardarán los outputs (correlativa, pares para etiquetar, reportes)
OUT_DIR = Path('/content/drive/MyDrive/rues_linker_outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Modo del pipeline (BALANCEADO recomendado para primera corrida)
MODE = 'BALANCEADO'

# Motor LSH. 'disk_based' es OBLIGATORIO para producción >1M registros.
# El default autoselecciona disk_based solo si n_records > 1M, pero lo forzamos
# aquí para asegurar el camino que correrá en producción real, sea cual sea el
# tamaño de la muestra inicial.
ENGINE_TYPE = 'disk_based'

# Cantidad de pares estratificados a generar para etiquetado
N_PARES_PARA_ETIQUETAR = 1500

## Paso 1 — Instalar la librería

In [ ]:
import subprocess, sys

# Extraer el tarball
!cd /content && tar -xzf {TARBALL}

# Instalar en modo desarrollo
REPO = Path('/content/rues-linker-v2.10.0')
!cd {REPO} && pip install -q -e ".[dev]"

# Verificar
import record_linkage
print(f'✅ Instalado record_linkage v{record_linkage.__version__}')

## Paso 2 — Inspeccionar la muestra real

Antes de procesar, verificamos qué tiene tu muestra: tamaño, columnas, NITs vacíos, distribución por fuente si aplica.

In [ ]:
import pandas as pd

if MUESTRA_REAL.suffix == '.parquet':
    df_real = pd.read_parquet(MUESTRA_REAL)
else:
    df_real = pd.read_csv(MUESTRA_REAL, dtype={COL_NIT: str})
df_real[COL_NIT] = df_real[COL_NIT].fillna('')

print(f'Registros: {len(df_real):,}')
print(f'Columnas: {list(df_real.columns)}')
print(f'NITs únicos: {df_real[COL_NIT].nunique():,}')
print(f'NITs vacíos: {(df_real[COL_NIT] == "").sum():,}')
print()
print('Distribución de longitud NIT:')
print(df_real[df_real[COL_NIT] != ''][COL_NIT].str.len().value_counts().sort_index())
print()
if 'FUENTE' in df_real.columns:
    print('Distribución por fuente:')
    print(df_real['FUENTE'].value_counts())

print()
print('Primeras 5 filas:')
df_real.head()

## Paso 3 — Correr v2.10.0 sobre la muestra completa

Esto puede tomar 5-30 minutos según el tamaño (50k regs ≈ 5 min, 200k ≈ 20 min, 1M ≈ 1-2 horas).

In [ ]:
import time
import tempfile
from record_linkage.deduplication.unified import (
    _build_deduplication_config,
    _prepare_for_deduplication,
)
from record_linkage.pipeline.linkage_pipeline import RecordLinkagePipeline

# Construir config forzando engine_type para que coincida con producción real.
config = _build_deduplication_config(
    'deduplication_standard', MODE, len(df_real)
)
config['linkage_engine_class'] = ENGINE_TYPE  # 'disk_based' obligatorio
print(f"Motor LSH activo: {config['linkage_engine_class']}")

t0 = time.perf_counter()

df_prep = _prepare_for_deduplication(
    df_real[[COL_NIT, COL_NAME]].copy(),
    COL_NIT, COL_NAME, MODE,
    remove_top_words=config['profiles'][config['profile']].get('remove_top_words', 20),
)
pipeline = RecordLinkagePipeline(config, profile='deduplication_standard')
with tempfile.TemporaryDirectory() as tmp:
    result = pipeline.run(
        sources={'DEDUP_SOURCE': df_prep},
        output_dir=tmp,
        source_priority=['DEDUP_SOURCE'],
        validate_data=False,
        generate_visualizations=False,
        show_progress=True,
    )
    corr = result.get('correlative_table').copy()

dt = time.perf_counter() - t0
print(f'\n✅ Pipeline completo en {dt:.1f}s ({len(df_real)/dt:.0f} regs/s)')
print(f'   {corr["ID_GRUPO"].nunique():,} grupos predichos')
print(f'   {len(corr):,} registros con grupo asignado')
stats = {}

## Paso 4 — Estadísticas de los clusters (sin necesidad de ground truth)

Esto te dice si los grupos predichos parecen razonables a primera vista. **Banderas rojas:**
- Pocos grupos (mucha fusión) → posible exceso de FP.
- Muchos grupos del tamaño exacto del número de registros (sin fusión) → posible exceso de FN.
- Un grupo gigantesco (1000+ registros) → bug de fusión catastrófica.

In [ ]:
corr = corr.sort_values('ORIGINAL_INDEX').reset_index(drop=True)

tam_grupo = corr.groupby('ID_GRUPO').size()
print(f'Distribución de tamaños de grupo:')
print(tam_grupo.describe().round(2))
print()
print(f'Singletons (grupos de tamaño 1): {(tam_grupo == 1).sum():,}')
print(f'Grupos pequeños (2-5): {((tam_grupo >= 2) & (tam_grupo <= 5)).sum():,}')
print(f'Grupos medianos (6-20): {((tam_grupo >= 6) & (tam_grupo <= 20)).sum():,}')
print(f'Grupos grandes (21+): {(tam_grupo >= 21).sum():,}')
print()
print('Top-10 grupos más grandes (sospechosos de sobre-fusión):')
top_grupos = tam_grupo.sort_values(ascending=False).head(10)
for gid, n in top_grupos.items():
    muestra_nombres = df_real.iloc[corr[corr['ID_GRUPO'] == gid].index][COL_NAME].head(5).tolist()
    print(f'  Grupo {gid}: {n} regs — ejemplos: {muestra_nombres}')

In [ ]:
# Guardar la correlativa para análisis offline
corr_out = OUT_DIR / 'correlativa_v2_10_0.csv'
df_correlativa_completa = df_real.copy()
df_correlativa_completa['ID_GRUPO_PREDICHO'] = corr['ID_GRUPO'].values
df_correlativa_completa.to_csv(corr_out, index=False)
print(f'✅ Correlativa guardada: {corr_out}')

## Paso 5 — Generar pares estratificados para etiquetado manual

Este es el paso clave del Sprint 1.1. Genera ~1500 pares **NO aleatorios**, estratificados en 5 categorías:

1. **alta_confianza** (score ≥ 0.90): verificar si hay FP sistemáticos en la zona "segura".
2. **frontera** (0.65 ≤ score < 0.72): zona donde se juega F1; etiquetar aquí permite recalibrar el threshold.
3. **nit_identico_nombre_disimil**: caso típico P4 del dataset robusto.
4. **nombre_alto_nit_distinto**: caso típico G (¿son entidades distintas o fusión legítima?).
5. **potencial_fn** (0.50 ≤ score < 0.65): pares cercanos al threshold que el sistema descarta.

El output es un CSV que abres en Excel/Sheets y llenas la columna `MISMO_GRUPO` con `True`/`False`.

In [ ]:
PARES_OUT = OUT_DIR / 'pares_para_etiquetar.csv'

!cd {REPO} && python scripts/generar_pares_para_etiquetar.py \
    --input {MUESTRA_REAL} \
    --col-nit {COL_NIT} \
    --col-name {COL_NAME} \
    --salida {PARES_OUT} \
    --n-pares {N_PARES_PARA_ETIQUETAR} \
    --mode {MODE}

print()
print(f'✅ {PARES_OUT}')
print()
print('SIGUIENTE PASO MANUAL:')
print('  1. Descarga el CSV de Drive a tu computador.')
print('  2. Abre en Excel o Sheets.')
print('  3. Llena la columna MISMO_GRUPO con True/False.')
print('  4. Opcionalmente, llena CASO_FRONTERA con el tipo (P1_, G_, etc.) y NOTAS.')
print('  5. Guarda como `pares_etiquetados.csv` en la misma carpeta de Drive.')
print('  6. Corre el bloque "Paso 6" cuando termines (próxima sesión).')

## Paso 6 — Medir contra etiquetas manuales (próxima sesión, tras etiquetar)

Después de etiquetar los pares manualmente en Excel y guardar como `pares_etiquetados.csv`, este paso produce el **primer F1 REAL** sobre tu producción.

In [ ]:
PARES_ETIQUETADOS = OUT_DIR / 'pares_etiquetados.csv'
REPORTE = OUT_DIR / 'reporte_calidad_v2_10_0.md'
ERRORES = OUT_DIR / 'errores_para_revision.csv'

if PARES_ETIQUETADOS.exists():
    !cd {REPO} && python scripts/medir_con_ground_truth.py \
        --input {MUESTRA_REAL} \
        --pares-etiquetados {PARES_ETIQUETADOS} \
        --col-nit {COL_NIT} \
        --col-name {COL_NAME} \
        --mode {MODE} \
        --reporte {REPORTE} \
        --errores-csv {ERRORES}
    print(f'\n📝 Reporte en: {REPORTE}')
    print(f'📝 Errores en: {ERRORES}')
else:
    print(f'⏸  No se encontró {PARES_ETIQUETADOS}.')
    print('   Termina de etiquetar y vuelve a correr este bloque.')

## Apéndice — Activar Fix #2 (opt-in tras calibración)

Si tras etiquetar y ver el reporte detectas FP por nombres genéricos ("INVERSIONES SAS × N"), activa el Fix #2 y re-mide. **NO actives sin primero etiquetar** — sobre corpus pequeños o sesgados puede empeorar el F1.

In [ ]:
# Activar Fix #2 modificando el perfil al vuelo
from record_linkage.deduplication.unified import _build_deduplication_config
from record_linkage.pipeline.linkage_pipeline import RecordLinkagePipeline
from record_linkage.deduplication.unified import _prepare_for_deduplication

config = _build_deduplication_config('deduplication_standard', MODE, len(df_real))
active = config['profile']
config['profiles'][active]['generic_name_penalty'] = 0.5
config['profiles'][active]['generic_name_top_n'] = 50
config['profiles'][active]['generic_name_max_tokens'] = 3
config['profiles'][active]['generic_name_min_sim'] = 0.85

df_prep = _prepare_for_deduplication(
    df_real[[COL_NIT, COL_NAME]].copy(), COL_NIT, COL_NAME, MODE,
    remove_top_words=20,
)
pipeline = RecordLinkagePipeline(config, profile='deduplication_standard')
with tempfile.TemporaryDirectory() as tmp:
    result = pipeline.run(
        sources={'DEDUP_SOURCE': df_prep},
        output_dir=tmp,
        source_priority=['DEDUP_SOURCE'],
        validate_data=False,
        generate_visualizations=False,
        show_progress=False,
    )
    corr_fix2 = result.get('correlative_table').copy()

corr_fix2 = corr_fix2.sort_values('ORIGINAL_INDEX').reset_index(drop=True)
print(f'Con Fix #2 ON: {corr_fix2["ID_GRUPO"].nunique():,} grupos predichos')
print(f'Sin Fix #2:    {corr["ID_GRUPO"].nunique():,} grupos predichos')
print(f'Δ grupos: {corr_fix2["ID_GRUPO"].nunique() - corr["ID_GRUPO"].nunique():+,} '
      f'(positivo = más grupos = menos fusión = menos FP esperados)')